# Lecture: Evaluating Language Models (EM, BLEU/ROUGE, judge, RAG)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain exact-match in one sentence and why it hides partial progress (§4).
- Compute clipped n-gram precision and the brevity penalty by hand on a toy pair (§5).
- Explain ROUGE as the recall-side mirror of BLEU's precision (§6).
- Explain rubric LLM-as-judge scoring and top-k retrieval, and read the base-vs-LoRA-vs-RAG split (§7–§9).


## The big idea

Grading a model is like judging a spelling test four ways at once. **Exact-match** is the strict teacher: whole answer letter-perfect or zero. **BLEU** is the careful editor: what share of the guess's short word-chains also appear in the answer (precision), with a **brevity penalty** multiplying down guesses that dodge words by staying short. **ROUGE** is the generous coach: what share of the *answer* the guess recovered (recall). And **LLM-as-judge** asks another model to rate 1–5 against a written rubric — a customer review instead of a ruler. The lab grades three students on reversing two words: the frozen **base** (never learned it), **LoRA** (rule stored in weights), **RAG** (rule assembled from retrieved examples pasted into the question). One score ties systems that differ; five scores split them.


In [ ]:
import numpy as np  # bring in numpy for the x slots
import matplotlib.pyplot as plt  # bring in pyplot for the grouped bars
import random  # bring in stdlib random for the seed habit
random.seed(15)  # fix stdlib seed so toy picks match each run
np.random.seed(15)  # fix numpy seed so the lecture matches each run
ref = 'cats sit down'  # toy answer every guess is compared against
labels = ['exact', 'partial', 'wrong']  # short names for the three guesses
em = [1.0, 0.0, 0.0]  # exact-match: only the perfect copy scores 1
overlap = [1.0, 0.67, 0.0]  # word-overlap share standing in for BLEU/ROUGE
print('ref:', ref)  # show the answer key used below
print('EM:', em)  # show strict scores: partial gets no credit
print('overlap:', overlap)  # show forgiving scores: partial gets 0.67
x = np.arange(3)  # x slots for the three guess types
w = 0.35  # bar half-width so paired bars sit side by side
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the pairs
ax.bar(x - w / 2, em, w, label='exact-match')  # left bars: strict 1-or-0 verdict
ax.bar(x + w / 2, overlap, w, label='overlap')  # right bars: partial credit view
ax.set_xticks(x, labels)  # name each guess type under its pair
ax.set_ylim(0, 1.15)  # headroom so the 1.0 bars are fully visible
ax.set_ylabel('score (higher is better)')  # label defines the score direction
ax.set_title('Strict vs forgiving grading')  # title in plain words
ax.legend()  # key telling which color is which metric
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the grouped bars
print('Lesson: EM hides progress; overlap reveals it.')  # one-sentence takeaway
print('RECALIBRATE: constants above are the author CPU run (SEED=15); paste your results/eval.csv + ragk.csv row to replace them - ordering/shape is the claim, digits are not')


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 is strict, §5–§6 forgive in opposite directions, §7–§8 judge and retrieve.

| Term | One-liner | Section |
|---|---|---|
| Exact-match (EM) | 1 if the answer matches letter-for-letter after lowercasing, else 0 | §4 |
| BLEU | Clipped n-gram precision × brevity penalty (punishes short guesses) | §5 |
| Brevity penalty | `exp(1 − ref_len/guess_len)` when the guess is short, else 1 | §5 |
| ROUGE-1 / ROUGE-L | Unigram overlap / longest-shared-run overlap, reported as F1 | §6 |
| LLM-as-judge | A 1–5 rubric score from another model (here a rule-based stand-in) | §7 |
| RAG / top-k | Paste k retrieved examples into the question; more demos, better coverage | §8 |


### §4 Exact-match — the strict teacher

**In one sentence: exact-match is 1 if the prediction equals the reference after lowercasing and trimming spaces, else 0 — no partial credit, ever.**

**Normalization** (lowercase, collapse spaces) keeps grading fair — “ Gnik ” still counts — but after that, one wrong letter is a zero. That harshness is a feature for reversal: a near-miss is a failed chore. Watch it: five toy answers where one slip (“thign” for “night”) drags the mean to 0.80 — and notice the trap, which §5–§6 fix: “thign” clearly *almost* worked, but EM cannot say so.


In [ ]:
import numpy as np  # reuse numpy for the mean
import matplotlib.pyplot as plt  # reuse pyplot for the strict bars
np.random.seed(15)  # fix the seed for reproducibility
refs = ['gnik tac', 'evol', 'night', 'nyad', 'olleh']  # five answer keys
preds = ['gnik tac', 'evol', 'thign', 'nyad', 'olleh']  # one one-letter slip hiding inside
em = [int(p.lower().strip() == r.lower().strip()) for p, r in zip(preds, refs)]  # normalize, then strict compare
print('per-answer EM:', em)  # print [1, 1, 0, 1, 1]: the slip gets zero
print('mean EM:', round(float(np.mean(em)), 2))  # print 0.80: one slip costs 0.20
fig, ax = plt.subplots(figsize=(7, 3.5))  # make one figure for the bars
ax.bar(['a1 ok', 'a2 ok', 'a3 slip', 'a4 ok', 'a5 ok'], em, color=['tab:green', 'tab:green', 'tab:red', 'tab:green', 'tab:green'])  # green = copy, red = slip
ax.set_ylim(0, 1.2)  # headroom so the 1.0 bars are fully visible
ax.set_ylabel('exact-match (1 or 0)')  # label defines the strictness
ax.set_title('EM: one slip costs 0.20 (mean 0.80)')  # title in printed numbers
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the bars

### §5 BLEU and the brevity penalty — clipped precision, shortness punished

**In one sentence: BLEU is the geometric mean of clipped 1- to 4-gram precisions (each guess chain counts at most as often as it appears in the answer) times a brevity penalty that multiplies down guesses shorter than the answer.**

**Clipping** stops cheating by repetition: guessing “cats cats cats” against “cats sit down” scores 1/3, not 3/3, because the answer holds “cats” once. The **brevity penalty** stops cheating by silence: a 2-word guess for a 3-word answer is multiplied by exp(1 − 3/2) ≈ 0.607 even with perfect precision. Watch it: “cats sit” earns unigram 1.0 and bigram 1.0 but BLEU only 0.607 (the penalty bites), “dogs run fast” earns 0.0 — plus the penalty curve showing length 1 → 0.05 while length ≥ 4 → 1.0.


In [ ]:
import numpy as np  # reuse numpy for the curve axis
import math  # bring in math for exp(): the brevity-penalty formula
import matplotlib.pyplot as plt  # reuse pyplot for the penalty curve
np.random.seed(15)  # fix the seed for reproducibility
pt = 'cats sit'.split()  # guess tokens: 2 words
rt = 'cats sit down'.split()  # answer tokens: 3 words
uni = 2 / 2  # unigram precision: both guess words appear (clipped)
bi = 1 / 1  # bigram precision: 'cats sit' is a real answer chain
bp = math.exp(1 - 3 / 2)  # brevity penalty: guess is shorter (2 vs 3)
bleu = round(bp * math.sqrt(uni * bi), 3)  # BLEU over existing orders: penalty x geometric mean
print('unigram P:', uni, '| bigram P:', bi)  # print 1.0, 1.0: perfect precision
print('brevity penalty:', round(bp, 3))  # print 0.607: shortness punished anyway
print('BLEU cats-sit:', bleu)  # print 0.607: precision was perfect, length was not
print('BLEU dogs-run-fast:', 0.0)  # print 0.0: zero unigram overlap zeroes everything
cs = [1, 2, 3, 4, 5]  # guess lengths against a 4-word answer (perfect precision assumed)
bps = [round(math.exp(1 - 4 / c), 3) if c < 4 else 1.0 for c in cs]  # penalty per length
print('penalty curve:', bps)  # print 0.05, 0.368, 0.717, 1.0, 1.0
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the curve
ax.plot(cs, bps, marker='o')  # draw the penalty climbing to 1.0
ax.set_xlabel('guess length (words)')  # label the cheating axis
ax.set_ylabel('brevity penalty (multiplier)')  # label defines the punishment
ax.set_title('Short guesses are multiplied down (0.05 -> 1.0)')  # title in printed numbers
ax.grid(True, alpha=0.3)  # add faint grid to read values easily
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the curve

### §6 ROUGE — the recall-side mirror

**In one sentence: where BLEU asks “how much of the guess was right” (precision), ROUGE asks “how much of the answer was recovered” (recall) — ROUGE-1 counts shared single words, ROUGE-L counts the longest shared word run, both reported as F1.**

Same toy, flipped question: “cats sit” vs “cats sit down” recovers 2 of 3 answer words (recall 0.67) with no wrong words (precision 1.0), and F1 balances them to 0.80. ROUGE-L agrees here because the shared run “cats sit” is length 2 — the whole guess. A guess can have great BLEU but poor ROUGE (short and perfect) or vice versa (long and rambling) — which is exactly why the harness reports both. Watch it: precision/recall/F1 bars for a good guess (F1 0.80) vs a wrong one (0.0).


In [ ]:
import numpy as np  # reuse numpy for the bar positions
import matplotlib.pyplot as plt  # reuse pyplot for the P/R/F1 bars
np.random.seed(15)  # fix the seed for reproducibility
p_good, r_good = 2 / 2, 2 / 3  # good guess: precision 1.0, recall 0.67
f_good = round(2 * p_good * r_good / (p_good + r_good), 2)  # F1 balances the two: 0.80
print('good: P', round(p_good, 2), 'R', round(r_good, 2), 'F1', f_good)  # print 1.0, 0.67, 0.80
print('wrong: P 0.0 R 0.0 F1 0.0')  # print zeros: 'dogs run fast' recovers nothing
x = np.arange(3)  # three slots: precision, recall, F1
w = 0.35  # bar half-width for side-by-side pairs
fig, ax = plt.subplots(figsize=(6, 4))  # make one figure for the pairs
ax.bar(x - w / 2, [1.0, 0.67, 0.80], w, label='cats sit (good)')  # left: high bars
ax.bar(x + w / 2, [0.0, 0.0, 0.0], w, label='dogs run fast (wrong)')  # right: flat zeros
ax.set_xticks(x, ['precision', 'recall', 'F1'])  # name each metric under its pair
ax.set_ylabel('score (higher is better)')  # label defines the direction
ax.set_title('ROUGE-1: good guess F1 0.80, wrong guess 0.0')  # title in printed numbers
ax.set_ylim(0, 1.15)  # headroom so the 1.0 bar is fully visible
ax.legend()  # show which color is which guess
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the bars

### §7 LLM-as-judge — a rubric, not a ruler

**In one sentence: LLM-as-judge prompts another model with a written rubric (“5 = exact, 4 = trivial slip, …, 1 = unrelated”) and records its 1–5 rating — human-style quality that still needs validating.**

A rubric turns vibes into procedure: each level describes observable facts (exact copy? right words, wrong order? one-letter slip? missing word? unrelated?), so ratings are repeatable. But a judge can be generous or biased — so the lab checks **agreement**: the share of examples where “judge ≥ 4” says the same thing as EM. Watch it: the rubric ladder plus 8 toy ratings agreeing with EM on 7 of 8 (87.5%) — one generous 4 on a wrong answer.


In [ ]:
import numpy as np  # reuse numpy for the agreement arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the rubric diagram
np.random.seed(15)  # fix the seed for reproducibility
em = np.array([1, 1, 0, 0, 1, 0, 1, 0])  # strict verdicts on 8 toy answers
judge = np.array([5, 5, 2, 4, 4, 2, 5, 1])  # rubric ratings: one generous 4 on item 4
agree = (judge >= 4) == (em == 1)  # do judge and EM call it the same way?
print('judge ratings:', judge.tolist())  # show the 1-5 spread
print('agreement:', agree.tolist())  # show 7 True, 1 False (the generous one)
print('agreement rate:', round(float(agree.mean() * 100), 1), '%')  # print 87.5%
fig, ax = plt.subplots(figsize=(7, 3))  # wide flat figure for the rubric ladder
ax.axis('off')  # hide axes: this is a diagram, not a plot
ax.set_xlim(0, 10)  # fix placement range
ax.set_ylim(0, 3)  # fix placement range
rungs = [('1\nunrelated', 1), ('2\nwrong words', 3), ('3\nright words,\nwrong order', 5), ('4\ntrivial slip', 7), ('5\nexact', 9)]  # rubric rungs
for name, xc in rungs:  # draw each rung box
    ax.text(xc, 1.5, name, ha='center', va='center', fontsize=8,  # centered rung label
            bbox=dict(boxstyle='round', facecolor='wheat' if xc < 8 else 'lightgreen'))  # green = counts as right
ax.set_title('Rubric ladder: >= 4 agrees with EM 87.5% of the time')  # title in printed numbers
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram

### §8 RAG and top-k — knowledge in context instead of weights

**In one sentence: RAG retrieves the k most similar training pairs (exact-word hits first, char-bigram overlap as backoff) and pastes them into the question, so a frozen model can copy the pattern instead of having learned it.**

Each demo covers the words it contains — one demo teaches “king → gnik” but leaves the second word unknown, so EM stays 0.0 at k=0 and k=1. At k=2 both words are usually covered (EM 0.542); at k=4 coverage is comfortable (EM 0.708) — still below LoRA's 1.0, because retrieval sometimes fetches the wrong neighbors. Watch it: the retrieve → compose → score flow plus the k-curve that climbs only once coverage does.


In [ ]:
import numpy as np  # reuse numpy for the sweep arrays
import matplotlib.pyplot as plt  # reuse pyplot for flow + curve
np.random.seed(15)  # fix the seed for reproducibility
ks = [0, 1, 2, 4]  # demo counts pasted by the retriever in the lab
emk = [0.0, 0.0, 0.542, 0.708]  # measured EM per k: coverage drives quality
print('top-k:', ks)  # show the retrieval depths tested
print('EM per k:', emk)  # show quality jumps once both words are covered
fig, ax = plt.subplots(1, 2, figsize=(10, 4))  # two panels: flow + curve
ax[0].axis('off')  # left is a diagram, not a plot
ax[0].set_xlim(0, 10)  # fix placement range
ax[0].set_ylim(0, 4)  # fix placement range
flow = [('query\n2 words', 1.2), ('retrieve\ntop-k', 4.0), ('compose\nreversed', 6.5), ('score\nEM etc', 8.8)]  # retrieve-then-read stages
for name, xc in flow:  # draw each stage box
    ax[0].text(xc, 2.0, name, ha='center', va='center', fontsize=8,  # centered stage label
               bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = pipeline
for xa, xb in [(2.2, 3.0), (5.0, 5.6), (7.5, 8.0)]:  # arrow gaps between stages
    ax[0].annotate('', xy=(xb, 2.0), xytext=(xa, 2.0), arrowprops=dict(arrowstyle='->'))  # forward arrows
ax[0].set_title('RAG: retrieve, then compose')  # left title
ax[1].plot(ks, emk, marker='o')  # right: EM climbs once coverage does
ax[1].set_xlabel('retrieved demos (k)')  # label defines k simply
ax[1].set_ylabel('exact-match (higher is better)')  # label defines EM direction
ax[1].set_title('More demos -> higher EM (0.0 -> 0.708)')  # title in printed numbers
ax[1].grid(True, alpha=0.3)  # add faint grid to read values easily
plt.tight_layout()  # stop titles and labels from overlapping
plt.show()  # display the figure

## §9 How it all fits together — the system map

One harness run pushes every system output through the same **metric suite**: EM (§4, strict), BLEU with its brevity penalty (§5, precision-side), ROUGE-1/L (§6, recall-side), and the rubric **judge** (§7). The suite then **splits** base vs LoRA vs RAG (§8): EM alone ties near-misses with nonsense, but the softer metrics and the judge separate “almost” from “nothing”. The map below pins each metric to its station; the trace scores one single answer all five ways.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
import random  # bring in stdlib random only to satisfy the seed habit (unused here)
random.seed(15)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the fan-out pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 5)  # fix placement range
ax.text(1.5, 2.5, 'outputs\nbase/LoRA/RAG', ha='center', va='center', fontsize=8,  # left: the three systems
        bbox=dict(boxstyle='round', facecolor='wheat'))  # wheat = systems under test
suite = [('EM §4', 4.0), ('BLEU §5', 3.2), ('ROUGE §6', 2.4), ('judge §7', 1.6)]  # metric suite lanes (names + heights)
for name, y in suite:  # draw each metric lane
    ax.text(6, y, name, ha='center', va='center', fontsize=8,  # centered metric label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = metric
    ax.annotate('', xy=(5.2, y), xytext=(2.6, 2.5), arrowprops=dict(arrowstyle='->'))  # arrow from outputs
    ax.annotate('', xy=(10.0, 2.5), xytext=(6.8, y), arrowprops=dict(arrowstyle='->'))  # arrow into the split
ax.text(11.5, 2.5, 'split:\nLoRA > RAG\n> base (§8)', ha='center', va='center', fontsize=8,  # right: verdict
        bbox=dict(boxstyle='round', facecolor='lightgreen'))  # green = verdict
ax.set_title('System map: outputs -> metric suite -> verdict')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map

In [ ]:
import numpy as np  # reuse numpy for the F1 arithmetic
import math  # bring in math for exp() in the BLEU line
np.random.seed(15)  # fix the seed for reproducibility
ref = ['gnik', 'tac']  # answer: two reversed words
pred = ['tac', 'gnik']  # guess: right words, wrong order (the revealing case)
em = int(pred == ref)  # strict verdict (§4)
uni = 2 / 2  # clipped unigram precision: both words appear once each
bi = 0 / 1  # bigram precision: 'tac gnik' is not an answer chain
bleu = 0.0 if min(uni, bi) == 0 else 1.0 * math.exp(sum([math.log(uni), math.log(bi)]) / 2)  # zero precision zeroes BLEU (§5)
r1 = 2 * 1.0 * 1.0 / (1.0 + 1.0)  # ROUGE-1 F1: same words, so 1.0 (§6)
rl = 2 * 0.5 * 0.5 / (0.5 + 0.5)  # ROUGE-L F1: longest run is 1 of 2, so 0.5 (§6)
print('EM:', em)  # print 0: order matters, no credit
print('BLEU:', bleu)  # print 0.0: one zero-precision order kills it
print('ROUGE-1:', round(r1, 2))  # print 1.0: every word recovered
print('ROUGE-L:', round(rl, 2))  # print 0.5: order broken, run halved
print('judge: 3 (right words, wrong order)')  # print the human-style read (§7)

## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the suite above, on two-word reversal with a compositional split (novel pairs of known words):

1. **Cell 1 (setup)** — seeds plus `os.makedirs('./results')`. Same `SEED = 15` habit as this lecture.
2. **Cell 2 (metric library)** — codes `exact_match` (§4), BLEU with clipped precision + brevity penalty (§5), ROUGE-1/L F1 (§6), each with a unit test — our toy numbers are those tests by hand.
3. **Cell 3 (task)** — poses two-word reversal so multi-word answers make BLEU/ROUGE meaningful.
4. **Cell 4 (systems)** — runs base (frozen zero-shot) vs LoRA-r16 (rule in weights) vs **RAG** (lexical retrieve + compose-the-facts reader, §8) — our verdict bars sketch its `eval.csv`.
5. **Cell 5 (sweeps + figure)** — varies top-k 0/1/2/4 (§8: EM 0.0 → 0.708) and checks judge-vs-EM agreement (§7) into `ragk.csv` and `eval.png`.
6. **Cell 6 (cleanup)** — swap the heuristic `judge()` for a real LLM call with `JUDGE_PROMPT`, keeping the agreement check.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **“tac gnik” for “gnik tac” scores EM 0.0, BLEU 0.0, ROUGE-1 1.0, ROUGE-L 0.5, judge 3. Which metric is “right”?**

*Answer: all of them — they measure different things (§9 trace). EM/BLEU punish the swapped order to zero (§4–§5), ROUGE-1 sees every word present (§6), ROUGE-L halves for the broken run, and the judge (§7) lands in the middle. That spread is why the harness keeps all five.*

2. **Why does RAG sit at EM 0.0 for k=1 but 0.708 for k=4?**

*Answer: each demo covers only its own words (§8). With k=1 the second query word is still unseen (EM 0), while k=4 usually retrieves both words so the reader can compose the full answer. Coverage, not cleverness, drives the curve.*

3. **If EM already ranks base < RAG < LoRA, why keep BLEU/ROUGE and the judge?**

*Answer: EM ties everything partially right at 0 (§4's trap). BLEU/ROUGE split “almost” from “nothing” (§5–§6: a swapped answer still scores ROUGE-1 1.0), and the 1–5 judge adds a human-style read validated by 87.5% agreement (§7).*
